# 30. Did unlearning shrink the weights, or scramble them?

Unlearning drives the generated weights toward their offsets. The 21 October measurement
found that 41 percent of channels still respond afterwards, at roughly a sixth of their
earlier strength, and 59 percent went silent. That tells us the weights got smaller. It does
not tell us whether the ones that survived still point the same way.

Those are two very different outcomes:

* **Same direction, smaller.** The pattern survived and only its size changed. Multiplying
  the weights back up should restore accuracy with no training. The task is recoverable, and
  a privacy claim based on this unlearning is weak.
* **Different direction.** The pattern is gone. Multiplying up would amplify something else.
  Nothing is recoverable by rescaling.

Run this before any rescaling experiment. If the direction did not survive, the rescaling
experiment is already answered and does not need to run.

**This notebook needs no GPU and no images.** The unlearning objective is a noise term plus a
weight-space preservation term, so it never looks at a dataset. One hundred steps of
weight-space optimisation runs on a CPU. Nothing is downloaded except the saved model.

**Why it unlearns rather than loading an unlearned model.** The study plan sets
`keep_post_unlearning_checkpoint` to false, so those models were deleted once their reports
were verified. A first version of this notebook looked for them and found none for any seed.
Notebook 27 has the same problem and solves it the same way: load the source and apply one
unlearning request in the session.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
# Importing uncle twice in one session gives two copies of the same classes, and a
# checkpoint saved by one will not load into the other. Stop before that happens.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-direction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
# boto3 talks to R2. tqdm is pulled in by uncle.telemetry.
for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle.paired_diagnostic import _forget_generated
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.measure import direction_change, relative_change, weight_size

import torch
# The unlearning loop uses a noise term and a weight-space preservation term. Neither reads
# an image, so a GPU would spend its time waiting. Forcing CPU also keeps the result
# comparable between a laptop and Colab.
DEVICE = "cpu"

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", DEVICE)

## 2. Settings

`FORGET_TASK` is the task to unlearn. `CONTROL_TASK` is a task the same model keeps. The
control matters: it says whether anything we see is specific to the forgotten task or
happened to the whole generator. The preservation term is supposed to hold the control
where it was, so its direction should stay at 1.0 throughout. If it does not, the reading
for the forgotten task cannot be trusted yet.

Both are the same pair the 21 October neuron measurement used, so the two measurements
describe the same event.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
SEEDS = [0, 1, 2]            # about an hour on CPU, roughly 19 minutes per seed

FORGET_TASK = "3"
CONTROL_TASK = "0"

# The study's own unlearning length. Notebook 27 used the same value, so the endpoint here
# should match the 43 percent size loss that notebook recorded.
BURN_IN = 100
INTERVAL = 5                 # readings are weight-only and nearly free, so take many

# A channel counts as dead when its weights have essentially no size left. Direction has no
# meaning for a zero vector, so these are counted separately rather than averaged in.
DEAD_NORM = 1e-8
# Below this cosine we call a channel's direction changed rather than preserved. 0.9 is
# about 26 degrees. It is a reporting threshold, not a statistical test.
PRESERVED_ABOVE = 0.9

# The same cache notebook 29 uses, so a source model downloaded by either is reused.
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
SAVE_TO_R2 = True

print("Forgetting task", FORGET_TASK, "| control task", CONTROL_TASK)
print(f"{BURN_IN} unlearning steps, reading every {INTERVAL}, seeds {SEEDS}")

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
print("Study:", plan["study_id"], "| order", ORDER, "learns", ORDER_TASKS)

## 4. What we measure

Two numbers, taken together, at every reading.

**Relative change** says how far the weights moved, as a fraction of their own earlier size.
`uncle/measure.py` already had this and notebook 27 reports it.

**Direction change** is new, in the same module. It is the cosine similarity between the
earlier and current weights: their dot product divided by both their lengths. It ignores
size completely.

* 1.0 means the same direction. The weights were scaled, not changed.
* 0.0 means unrelated directions.
* -1.0 means the opposite direction.

Taken alone, neither answers the question. A large relative change happens whether the
weights shrank or scrambled. The pair separates those two cases.

We subtract each tensor's offset first. A generated weight is a raw generator output times a
fixed scale, plus a fixed offset. BatchNorm scale weights have an offset of 1.0, so a weight
that collapsed entirely to its offset would still look like a perfect direction match if the
constant were left in. Subtracting it leaves the part unlearning acts on.

In [ ]:
def generated_part(hypernet, task):
    """One task's generated weights with each tensor's fixed offset removed."""
    return {name: value.detach() - hypernet.offsets[name]
            for name, value in hypernet.weights_for(task).items()}


def channel_report(reference, current):
    """The direction question asked separately for each output channel.

    Row 0 of a weight tensor is one output channel: one convolution filter, or one row of the
    classifier. Flattening everything after dimension 0 gives one vector per channel. A
    whole-tensor cosine averages over all channels at once, so a few preserved channels can
    hide many scrambled ones, and the 21 October silent-channel finding is a per-channel fact.

    Tensors with a single dimension, such as biases, have no channel structure and return
    None. The pooled cosine covers them.
    """
    if reference.dim() < 2:
        return None
    before = reference.double().reshape(reference.shape[0], -1)
    after = current.double().reshape(current.shape[0], -1)
    before_size, after_size = before.norm(dim=1), after.norm(dim=1)
    alive = (before_size > DEAD_NORM) & (after_size > DEAD_NORM)
    cosines = (before[alive] * after[alive]).sum(dim=1) / (before_size[alive] * after_size[alive])
    return {
        "channels": int(reference.shape[0]),
        # Channels that had size before and have none now. Rescaling cannot bring these
        # back: any multiplier times zero is still zero.
        "died": int(((before_size > DEAD_NORM) & (after_size <= DEAD_NORM)).sum()),
        "measurable": int(alive.sum()),
        "median_cosine": float(cosines.median()) if cosines.numel() else float("nan"),
        "lowest_cosine": float(cosines.min()) if cosines.numel() else float("nan"),
        "preserved": int((cosines >= PRESERVED_ABOVE).sum()),
        # How much smaller each surviving channel became. If unlearning is a pure scaling,
        # this is the factor a rescaling experiment would multiply by.
        "median_size_ratio": float((after_size[alive] / before_size[alive]).median())
                             if alive.any() else float("nan"),
    }

## 5. Rebuild the model and take the reading before anything changes

`UnCLe` accepts no dataset here. Its own docstring says target and tasks are needed only to
measure accuracy and to learn, and a caller that just wants `forget` can leave them out. We
pass the target, because the running statistics template comes from it, and leave the tasks
out, which is what makes this notebook free of any dataset.

The noise the unlearning uses comes from its own generator, seeded from the run's seed, the
word forget and the task name. It does not depend on the global random state, so this run is
repeatable without restoring the saved random state.

In [ ]:
def source_job(seed):
    return next(job for job in ALL_JOBS if job["id"] == f"sources/{ORDER}/seed_{seed}")


def prepare(seed):
    """The finished eight-task model for one seed, ready to unlearn from."""
    state = storage.load(source_job(seed))
    if state is None:
        raise RuntimeError(f"No source model in R2 for seed {seed}.")
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", t) for t in ORDER_TASKS)
                                   + (("forget", FORGET_TASK),),
                       "burn_in": BURN_IN})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, None, progress=False)
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    # Forgetting protects every learned task except its own target, as the study does.
    protected = [task for task in state["seen"] if task != FORGET_TASK]
    if CONTROL_TASK not in protected:
        raise ValueError(f"Control task {CONTROL_TASK} is not among the protected tasks.")
    return config, hypernet, trainer, protected, list(state["seen"])

## 6. Run the unlearning and watch both numbers

A reading every 20 steps gives the shape of the change, not just its endpoint. That matters
here: if the direction holds at 1.0 the whole way down while the size falls, unlearning is a
scaling and nothing else. If the direction slips away gradually, it is not.

In [ ]:
runs = []
for seed in SEEDS:
    print(f"\nseed {seed}: loading the source model")
    config, hypernet, trainer, protected, seen = prepare(seed)
    tracked = {FORGET_TASK: "forgotten", CONTROL_TASK: "control"}
    # The state to compare everything against, captured before a single step runs.
    reference = {task: generated_part(hypernet, task) for task in tracked}
    rows = []

    def reading(step):
        row = {"step": step}
        for task, role in tracked.items():
            now = generated_part(hypernet, task)
            row[role] = {
                "relative_change": relative_change(now, reference[task]),
                "direction": direction_change(now, reference[task]),
                "raw_weight_size": weight_size(hypernet, task),
            }
        rows.append(row)
        return row

    # Step 0 must read exactly 0.0 change and 1.0 direction. It is comparing the reference
    # with itself, so anything else means the reference was captured wrongly.
    first = reading(0)
    assert abs(first["forgotten"]["relative_change"]) < 1e-9, "Reference captured after a change."

    began = time.perf_counter()

    def on_step(record):
        step = record["step"]
        if step > 0 and step % INTERVAL == 0:
            row = reading(step)
            rate = (time.perf_counter() - began) / step
            print(f"  step {step:>4}: forgotten direction {row['forgotten']['direction']:.4f}, "
                  f"size change {row['forgotten']['relative_change']:.4f}, "
                  f"control direction {row['control']['direction']:.4f}  "
                  f"({rate:.2f}s/step)")

    _forget_generated(trainer, FORGET_TASK, protected, config.burn_in, on_step)
    if rows[-1]["step"] != config.burn_in:
        reading(config.burn_in)
    elapsed = (time.perf_counter() - began) / 60

    # The per-channel picture, once, at the end.
    channels = {}
    for task, role in tracked.items():
        now = generated_part(hypernet, task)
        channels[role] = {name: channel_report(reference[task][name], now[name])
                          for name in sorted(reference[task])}

    runs.append({"seed": seed, "seen": seen, "protected": protected,
                 "burn_in": config.burn_in, "minutes": elapsed,
                 "rows": rows, "channels": channels})
    print(f"  done in {elapsed:.1f} minutes")
    del hypernet, trainer, reference

RESULT_PATH = CACHE / f"direction_{ORDER}_forget{FORGET_TASK}.json"
RESULT_PATH.write_text(json.dumps(
    {"head": HEAD, "study": STUDY_ID, "order": ORDER, "device": DEVICE,
     "forget_task": FORGET_TASK, "control_task": CONTROL_TASK, "runs": runs},
    indent=2), encoding="utf-8")
print("\nSaved", RESULT_PATH)
if SAVE_TO_R2:
    key = f"uncle/unlearning_direction/{STUDY_ID}/{ORDER}/forget{FORGET_TASK}.json"
    # overwrite, because rerunning should replace this notebook's own result rather than stop.
    store.upload(RESULT_PATH, key, overwrite=True)
    print("Uploaded", key)

## 7. The trajectory

One block per seed. Read the forgotten columns against the control columns on the same row.

In [ ]:
for run in runs:
    print(f"\nseed {run['seed']}, forgetting task {FORGET_TASK}, "
          f"{run['burn_in']} steps in {run['minutes']:.1f} minutes")
    print(f"{'step':>5} | {'fgt dir':>8} {'fgt size chg':>13} {'fgt raw size':>13} "
          f"| {'ctl dir':>8} {'ctl size chg':>13}")
    for row in run["rows"]:
        print(f"{row['step']:>5} | {row['forgotten']['direction']:>8.4f} "
              f"{row['forgotten']['relative_change']:>13.4f} "
              f"{row['forgotten']['raw_weight_size']:>13.4f} | "
              f"{row['control']['direction']:>8.4f} "
              f"{row['control']['relative_change']:>13.4f}")

    print("\n  per channel at the end, forgotten task:")
    print(f"  {'tensor':<34} {'chans':>6} {'died':>6} {'med cos':>9} {'low cos':>9} {'size x':>8}")
    for name, report in run["channels"]["forgotten"].items():
        if report is None:
            continue
        print(f"  {name[:34]:<34} {report['channels']:>6} {report['died']:>6} "
              f"{report['median_cosine']:>9.4f} {report['lowest_cosine']:>9.4f} "
              f"{report['median_size_ratio']:>8.4f}")

## What the numbers mean

**Check the control first.** Its direction should stay at 1.0000 and its size change near
0.0000 for every step. The preservation term exists to hold the protected tasks still. If the
control drifts, unlearning is spilling onto tasks it was supposed to leave alone, which is a
finding in itself and also means the forgotten column needs care.

**Forgotten task, direction stays near 1.0 while size change grows.** Unlearning is a scaling
and nothing else. The pattern is intact and quiet. This supports running the rescaling
experiment, and it weakens any privacy claim for this unlearning, because the information is
still in the model.

**Forgotten task, direction falls away.** The pattern changed, not just its size. Rescaling
would amplify something different from the original weights, so that experiment is answered
and does not need GPU time. Where the direction starts falling, against where the size
starts dropping, says whether the two happen together or one follows the other.

**The died column.** Channels that had size at the start and have none at the end. Rescaling
cannot recover these. If this number is large while the surviving cosines stay high,
unlearning did two separate things: it removed some channels and turned others down. Those
need separate conclusions, and only the second is undone by a multiplier.

**size x, the median size ratio.** If unlearning is close to a pure scaling, this is the
factor a rescaling experiment should use, per tensor. Compare it with the per-stage values
from 21 October: 0.29 at the stem, then 0.14, 0.13, 0.16 and 0.19. Those came from channel
responses to images, not from weights, so agreement is a check and a disagreement is worth
understanding before rescaling anything.

**Endpoint check against notebook 27.** That notebook recorded the generated weights losing
about 43 percent of their size over the same 100 steps. The raw size column here should land
in the same region. A large gap means these two runs are not doing the same thing.

## What this notebook does not do

It does not measure effective rank, which needs a forward pass over images. It does not test
recovery; it only says whether a recovery attempt by rescaling is worth running. It runs one
seed by default, and one seed is an observation rather than a result. Add seeds 1 and 2 once
the first run's time is known, and say how many were used when reporting anything.